# 🏗️ Chapter 10: Infrastructure as Code

📖 Book: Chapter 10, pp. 392–429 · ⏱️ 2–3 hours · 🧱 Needs: Ch 2, 9 · Level: Intermediate

---

> **🎯 Key idea:** Declarative IaC (Terraform/OpenTofu HCL or Pulumi Python) computes state diffs before applying, eliminating manual console drift.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** TF 0.11 `"${var.x}"` + inline S3 `acl`  
> **🆕 Now (2026):** HCL2 `var.x` + `aws_s3_bucket_public_access_block`  
> **Why:** Cleaner syntax & secure-by-default S3

> **📜 Then (book, ~2020):** Terraform + Pulumi v1/v2  
> **🆕 Now (2026):** Terraform / OpenTofu + Pulumi v3+  
> **Why:** Open-source fork & typed Pulumi args

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
Someone clicked a checkbox in the AWS Console to open an S3 bucket for 'just five minutes,' and now production differs from your docs. Manual cloud setup is slow and impossible to audit or reproduce cleanly. Declarative IaC (Terraform/OpenTofu and Pulumi) lets you preview diffs, detect drift, and enforce security policies in code.

### Concept: Declarative State & Drift
**What:** Desired blueprint vs. live infra state  
**Why:** Console clicks cause undocumented drift  
**How:** IaC engines diff state and plan minimal changes  
**When:** All cloud infrastructure

📖 *Book listing:* `devops4all.dev` Modular Terraform HCL (pp. 401–412)

**Plain-language analogy:** Imperative scripts are turn-by-turn driving directions (they break if you're already halfway there); declarative IaC is setting a GPS destination—the engine figures out the diff from where you are.

**Walkthrough:** The book organizes S3, ACM (`us-east-1`), CloudFront, and Route 53 into `main.tf`, `variables.tf`, and `outputs.tf` modules, running `init -> plan -> apply` cleanly.

### Concept: Pulumi Projects vs. Stacks
**What:** Python IaC code vs. env deployments  
**Why:** Use loops, functions & tests in real Python  
**How:** One project -> multiple stacks (`staging`, `prod`)  
**When:** Teams preferring Python over HCL

📖 *Book listing:* Pulumi `aws-python` Stack (pp. 413–428)

**Plain-language analogy:** Pulumi projects vs. stacks are like a house blueprint (project) built in two separate neighborhoods (`staging` and `prod` stacks).

**Walkthrough:** The book ports the same static website stack to Python (`pulumi_aws`), isolating `staging` and `prod` state into separate stacks while noting `snake_case` vs. `camelCase` args.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Generate Terraform .tfvars.json
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Generate Terraform .tfvars.json
import json, tempfile
from pathlib import Path

def write_tfvars(path: Path, region: str, env: str, tags: dict[str, str]) -> dict:
    payload = {'aws_region': region, 'environment': env, 'common_tags': tags}
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    return payload

with tempfile.TemporaryDirectory() as tmp:
    print(write_tfvars(Path(tmp) / 'prod.tfvars.json', 'us-east-1', 'prod', {'Owner': 'devops'}))

{'aws_region': 'us-east-1', 'environment': 'prod', 'common_tags': {'Owner': 'devops'}}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: Required resource tag validator
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: Required resource tag validator
def missing_tags(tags: dict[str, str]) -> set[str]:
    return {'env', 'owner', 'cost_center'} - set(tags.keys())

print('Missing tags:', missing_tags({'env': 'prod'}))

Missing tags: {'cost_center', 'owner'}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Parse Pulumi/IaC YAML config with yaml.safe_load
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Parse Pulumi/IaC YAML config with yaml.safe_load
PULUMI_YAML = '''
name: static-site
runtime: python
config:
  aws:region: us-east-1
'''
try:
    import yaml
    print('Parsed Pulumi.yaml:', yaml.safe_load(PULUMI_YAML))
except ImportError:
    print('PyYAML not installed')

Parsed Pulumi.yaml: {'name': 'static-site', 'runtime': 'python', 'config': {'aws:region': 'us-east-1'}}


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

## 7. 🟡 Intermediate Examples: State Drift Detector & Pulumi Python SDK

#### Example 4: Intermediate Example 1: Pure-Python state drift detector
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Pure-Python state drift detector
def detect_drift(desired: dict, actual: dict) -> dict:
    return {
        'missing': [k for k in desired if k not in actual],
        'extra': [k for k in actual if k not in desired],
        'changed': {k: {'desired': desired[k], 'actual': actual[k]} for k in desired if k in actual and desired[k] != actual[k]},
    }

print('Drift:', detect_drift({'encrypted': True, 'public': False}, {'encrypted': True, 'public': True}))

Drift: {'missing': [], 'extra': [], 'changed': {'public': {'desired': False, 'actual': True}}}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

> **⚠️ Warning:** Cloud commands incur costs—clean up test resources afterward.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): Pulumi Python AWS S3
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): Pulumi Python AWS S3
# Requires: pip install pulumi pulumi-aws
# WARNING: Provisioning real AWS resources incurs charges. Always run `pulumi destroy` afterward!
try:
    import pulumi
    print('pulumi SDK available:', pulumi.__version__)
except ImportError:
    simulated_stack = {'resource': 'aws:s3:Bucket', 'name': 'site-bucket', 'acl': 'private'}
    print('pulumi not installed; simulated stack output:', simulated_stack)

pulumi not installed; simulated stack output: {'resource': 'aws:s3:Bucket', 'name': 'site-bucket', 'acl': 'private'}


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Pre-Apply Policy Compliance Gate**—blocks unencrypted or public S3 buckets before `apply`.

#### Example 6: def check_iac_compliance(resources: list[dict]) -> list[str]:
*Expected output:* Printed labelled output below.

In [6]:
def check_iac_compliance(resources: list[dict]) -> list[str]:
    violations: list[str] = []
    for res in resources:
        if res['type'] == 's3_bucket':
            if not res.get('encrypted', False):
                violations.append(f"{res['name']}: encryption disabled")
            if res.get('public', False):
                violations.append(f"{res['name']}: public access enabled")
    return violations

print('Violations:', check_iac_compliance([{'type': 's3_bucket', 'name': 'logs', 'encrypted': False, 'public': False}]))

Violations: ['logs: encryption disabled']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 7: Real Tool Demo: terraform / tofu validate in a temp dir (guarded with shutil.which)
*Expected output:* Printed labelled output below.

In [7]:
# Real Tool Demo: terraform / tofu validate in a temp dir (guarded with shutil.which)
import shutil, subprocess, tempfile
from pathlib import Path
iac_bin = shutil.which('tofu') or shutil.which('terraform')
if iac_bin:
    with tempfile.TemporaryDirectory() as tmp:
        (Path(tmp) / 'main.tf').write_text('variable "env" { default = "dev" }\n', encoding='utf-8')
        subprocess.run([iac_bin, 'init', '-backend=false'], cwd=tmp, capture_output=True, check=True)
        res = subprocess.run([iac_bin, 'validate'], cwd=tmp, capture_output=True, text=True, check=True)
        print(f'{Path(iac_bin).name} validate:', res.stdout.strip())
else:
    print('terraform/tofu not installed; skipped')

tofu validate: Success! The configuration is valid.


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Manual console edits** -> State drift on next `apply`. ✅ Make all changes in IaC code.
- ❌ **Destroying non-empty S3** -> `BucketNotEmpty` error. ✅ Empty bucket or set `force_destroy`.
- ❌ **ACM cert outside `us-east-1`** -> CloudFront rejects certificate. ✅ Provision CloudFront ACM in `us-east-1`.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement to_camel and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [8]:
# Starter Exercise Cell (implement to_camel and run to verify!)
def _run_starter_checks():
    assert to_camel('resource_record_name') == 'resourceRecordName'
def to_camel(snake: str) -> str:
    raise NotImplementedError('Implement to_camel')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement to_camel


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** How does declarative IaC differ from Boto3 scripts?

<details>
<summary>✅ Answer 1</summary>

Declarative IaC specifies desired state and computes diffs; Boto3 runs imperative steps.

</details>

**Q2:** What is state drift?

<details>
<summary>✅ Answer 2</summary>

Divergence between IaC state and live cloud resources, usually from manual console edits.

</details>

**Q3:** In Pulumi, what is a project vs. a stack?

<details>
<summary>✅ Answer 3</summary>

A project is the codebase; a stack is an isolated environment (`staging`, `prod`).

</details>

**Q4:** Why run `terraform plan` before `terraform apply`?

<details>
<summary>✅ Answer 4</summary>

To preview creations, updates, and destructive replacements before mutating live infra.

</details>

**Q5:** Which AWS region is required for CloudFront ACM certificates?

<details>
<summary>✅ Answer 5</summary>

`us-east-1`.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 2**: Split on `'_'` and capitalize parts[1:].

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [9]:
# Complete Solutions + Assert Verification
import json

def render_backend_tf(bucket: str, key: str, region: str) -> str:
    return f'terraform {{\n  backend "s3" {{\n    bucket = "{bucket}"\n    key = "{key}"\n    region = "{region}"\n  }}\n}}'

def to_camel(snake: str) -> str:
    parts = snake.split('_')
    return parts[0] + ''.join(p.capitalize() for p in parts[1:])

def cfn_s3_template(bucket_name: str) -> str:
    return json.dumps({'AWSTemplateFormatVersion': '2010-09-09', 'Resources': {'Site': {'Type': 'AWS::S3::Bucket', 'Properties': {'BucketName': bucket_name}}}})

def has_drift(desired: dict, actual: dict) -> bool:
    d = detect_drift(desired, actual)
    return bool(d['missing'] or d['extra'] or d['changed'])

def evaluate_plan(desired_resources: list[dict], live_map: dict) -> dict:
    violations = check_iac_compliance(desired_resources)
    drift = detect_drift({r['name']: r for r in desired_resources}, live_map)
    return {'approved': len(violations) == 0, 'violations': violations, 'drift': drift}

assert 'backend "s3"' in render_backend_tf('b', 'k', 'us-east-1')
assert to_camel('resource_record_name') == 'resourceRecordName'
assert json.loads(cfn_s3_template('b1'))['Resources']['Site']['Type'] == 'AWS::S3::Bucket'
assert has_drift({'a': 1}, {'a': 2}) is True
assert evaluate_plan([{'type': 's3_bucket', 'name': 'b1', 'encrypted': True, 'public': False}], {})['approved'] is True
print('All Chapter 10 assertions PASSED!')

All Chapter 10 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `tofu validate` / `terraform validate` — Syntax & config check (e.g. `tofu init -backend=false && tofu validate`)
- `*.tfvars.json` — Machine-generated TF vars (e.g. `json.dumps({'aws_region': 'us-east-1'})`)
- `pulumi up` / `pulumi destroy` — Deploy/teardown stack (e.g. `pulumi stack output`)

### ✅ What You Can Now Do
- Generate Terraform `.tfvars.json` and CloudFormation JSON templates from Python.
- Detect state drift between desired and live infrastructure dicts.
- Enforce pre-apply security policies (encryption, private S3) and validate HCL with `tofu`/`terraform validate`.

### ➡️ Next Chapter
Chapter 11 builds on this by packaging applications and backing services into Docker images and Compose v2 stacks.